# DMM (Digital Multimeter) Driver Test Notebook

| Layer | Class | Key Methods |
|---|---|---|
| 1 | `Instrument` | `idn()` |
| 2 | `Scpi` | `reset()`, `clear()`, `error()`, `wait()`, `self_test()`, `operation_complete()`, `initialize()` |
| 3 | `DMM` | Sense configuration, measurement reads |

**Instructions:** Run each cell sequentially. Verify the **Expected Result** after each cell. A known voltage/current/resistance source is recommended.

---
## 0. Setup & Connection

In [ ]:
from piec.drivers.autodetect import autodetect

In [ ]:
# --- Option A: Auto-detect the DMM ---
dmm = autodetect("dmm", verbose=True)

# --- Option B: Manually specify ---
# from piec.drivers.dmm.agilent_34410a import Agilent34410A
# dmm = Agilent34410A("GPIB0::22::INSTR")

✅ **PASS** if no error is raised.

---
## 1. Instrument & SCPI Tests

### 1.1 Identification (`idn`)

In [ ]:
idn_response = dmm.idn()
print("IDN Response:")
print(idn_response)

**Expected Result:** `Manufacturer, Model, Serial Number, Firmware Version`

✅ **PASS** if the string matches your physical instrument.

### 1.2 Reset (`reset`)

In [ ]:
dmm.reset()
print("Reset command sent.")

**Expected Result:** DMM returns to factory defaults (typically DC Voltage, autorange).

✅ **PASS** if the DMM display resets.

### 1.3 Clear Status (`clear`)

In [ ]:
dmm.clear()
print("Clear command sent.")

✅ **PASS** if no error is raised.

### 1.4 Error Query (`error`)

In [ ]:
error_response = dmm.error()
print("Error Status Register:", error_response)

**Expected Result:** Returns `0` (no errors). ✅ **PASS** if `0`.

### 1.5 Self Test (`self_test`)

In [ ]:
self_test_result = dmm.self_test()
print("Self-test result:", self_test_result)

**Expected Result:** Returns `0` (passed). May take several seconds. ✅ **PASS** if `0`.

### 1.6 Wait & Operation Complete

In [ ]:
dmm.wait()
opc_result = dmm.operation_complete()
print("Operation Complete:", opc_result)

**Expected Result:** Returns `1`. ✅ **PASS** if `1`.

### 1.7 Initialize (`initialize`)

In [ ]:
dmm.initialize()
print("Initialize command sent (reset + clear).")

✅ **PASS** if the DMM resets and no error is raised.

---
## 2. Measurement Configuration Tests

### 2.1 Set Sense Function (`set_sense_function`)

In [ ]:
dmm.set_sense_function(sense_func='VOLT')
print("Sense function set to VOLT.")

**Expected Result:** DMM display shows **Voltage** mode. ✅ **PASS** if correct.

In [ ]:
dmm.set_sense_function(sense_func='CURR')
print("Sense function set to CURR.")

**Expected Result:** DMM display shows **Current** mode. ✅ **PASS** if correct.

In [ ]:
dmm.set_sense_function(sense_func='RES')
print("Sense function set to RES.")

**Expected Result:** DMM display shows **Resistance** (Ω) mode. ✅ **PASS** if correct.

### 2.2 Set Measurement Coupling (`set_measurement_coupling`)

In [ ]:
dmm.set_sense_function(sense_func='VOLT')
dmm.set_measurement_coupling(coupling='DC')
print("Coupling set to DC.")

**Expected Result:** Display shows **DC V**. ✅ **PASS** if correct.

In [ ]:
dmm.set_measurement_coupling(coupling='AC')
print("Coupling set to AC.")

**Expected Result:** Display shows **AC V**. ✅ **PASS** if correct.

### 2.3 Set Sense Mode (`set_sense_mode`)

In [ ]:
dmm.set_sense_mode(sense_mode='2W')
print("Sense mode set to 2-wire.")

✅ **PASS** if 2-wire mode is active.

In [ ]:
dmm.set_sense_mode(sense_mode='4W')
print("Sense mode set to 4-wire.")

✅ **PASS** if 4-wire indicator is shown.

### 2.4 Set Sense Range (`set_sense_range`)

In [ ]:
dmm.set_sense_function(sense_func='VOLT')
dmm.set_measurement_coupling(coupling='DC')
dmm.set_sense_range(auto=True)
print("Sense range set to AUTO.")

✅ **PASS** if autorange is active.

In [ ]:
dmm.set_sense_range(range_val=10, auto=False)
print("Sense range set to 10 V (fixed).")

**Expected Result:** Fixed **10 V** range. ✅ **PASS** if range matches.

### 2.5 Set Integration Time (`set_integration_time`)

In [ ]:
dmm.set_integration_time(nplc=1)
print("Integration time: 1 NPLC.")

✅ **PASS** if no error is raised.

In [ ]:
dmm.set_integration_time(nplc=10)
print("Integration time: 10 NPLC (high accuracy).")

✅ **PASS** if no error is raised. Readings should be more stable.

---
## 3. Measurement (Read) Tests

⚠️ Connect a known reference to the DMM inputs for verification.

### 3.1 Quick Read (`quick_read`)

In [ ]:
reading = dmm.quick_read()
print(f"Quick Read: {reading}")

**Expected Result:** A **float** value in the currently configured function. ✅ **PASS** if reasonable numeric.

### 3.2 Get Voltage (`get_voltage`)

In [ ]:
dc_voltage = dmm.get_voltage(ac=False)
print(f"DC Voltage: {dc_voltage} V")

✅ **PASS** if a reasonable DC voltage value is returned.

In [ ]:
ac_voltage = dmm.get_voltage(ac=True)
print(f"AC Voltage: {ac_voltage} V")

✅ **PASS** if a reasonable AC RMS voltage value is returned.

### 3.3 Get Current (`get_current`)

In [ ]:
# Ensure leads are in current terminals!
dc_current = dmm.get_current(ac=False)
print(f"DC Current: {dc_current} A")

⚠️ Ensure leads are in the current input terminals.

✅ **PASS** if a reasonable current value is returned.

### 3.4 Get Resistance (`get_resistance`)

In [ ]:
resistance_2w = dmm.get_resistance(four_wire=False)
print(f"2-Wire Resistance: {resistance_2w} Ω")

✅ **PASS** if a reasonable resistance value is returned.

In [ ]:
resistance_4w = dmm.get_resistance(four_wire=True)
print(f"4-Wire Resistance: {resistance_4w} Ω")

✅ **PASS** if a reasonable 4-wire resistance value is returned.

---
## 4. Cleanup

In [ ]:
dmm.reset()
print("DMM reset to factory defaults.")

---
## Test Summary

| # | Method(s) Tested | Pass/Fail |
|---|------------------|-----------|
| 0 | `__init__` / `autodetect` | |
| 1.1 | `idn()` | |
| 1.2 | `reset()` | |
| 1.3 | `clear()` | |
| 1.4 | `error()` | |
| 1.5 | `self_test()` | |
| 1.6 | `wait()`, `operation_complete()` | |
| 1.7 | `initialize()` | |
| 2.1 | `set_sense_function()` | |
| 2.2 | `set_measurement_coupling()` | |
| 2.3 | `set_sense_mode()` | |
| 2.4 | `set_sense_range()` | |
| 2.5 | `set_integration_time()` | |
| 3.1 | `quick_read()` | |
| 3.2 | `get_voltage()` | |
| 3.3 | `get_current()` | |
| 3.4 | `get_resistance()` | |
| 4 | `reset()` | |

**Technician Signature:** _______________  
**Date:** _______________  
**Instrument Serial #:** _______________